In [1]:
import pandas as pd
import numpy as np

import choix
import statsmodels.api as sm

from sklearn.metrics import accuracy_score, log_loss

In [2]:
df = pd.read_csv('./data/bt_games_cleaned.csv')

context_cols = ["home_b2b", "away_b2b","home_rest_days", "away_rest_days", "rest_diff"]
feature_cols = ["diff_recent_margin","diff_recent_win_pct","diff_pts_pg_to_date","split_win_pct_delta"]

df_train=df[df["split"]=="train"].copy()
df_test=df[df["split"]=="test"].copy()

In [3]:
all_teams = sorted(
    pd.unique(
        np.concatenate([df["home_abbr"].values, df["away_abbr"].values])
    )
)
team_to_idx = {t: i for i, t in enumerate(all_teams)}

In [4]:
num_teams = len(all_teams)

comparisons = []
for _, row in df_train.iterrows():
    home = team_to_idx[row["home_abbr"]]
    away = team_to_idx[row["away_abbr"]]
    if row["home_win"] == 1:
        comparisons.append((home, away))
    else:
        comparisons.append((away, home))

s_hat = choix.mm_pairwise(
    n_items=num_teams,
    data=comparisons,
    alpha=1e-5,
    max_iter=20000,
    tol=1e-5
)
s_hat -= np.mean(s_hat)

In [5]:
def predict_bt(df, s, team_to_idx):
    """Predict home‑win probabilities via σ(s_home − s_away)."""
    logits = np.array([
        s[team_to_idx[h]] - s[team_to_idx[a]]
        for h, a in zip(df.home_abbr, df.away_abbr)
    ])
    return 1 / (1 + np.exp(-logits))


# ==== Train metrics ====
y_train = df_train.home_win.astype(int).values
p_train = predict_bt(df_train, s_hat, team_to_idx)

train_acc = accuracy_score(y_train, (p_train >= 0.5).astype(int))
train_ll  = log_loss(y_train, p_train)


# ==== Test metrics ====
y_test = df_test.home_win.astype(int).values
p_test  = predict_bt(df_test, s_hat, team_to_idx)

test_acc = accuracy_score(y_test, (p_test >= 0.5).astype(int))
test_ll  = log_loss(y_test, p_test)


# ==== Summary ====
print(f"Train accuracy: {train_acc:.4f} | Train log‑loss: {train_ll:.4f}")
print(f"Test  accuracy: {test_acc:.4f} | Test  log‑loss: {test_ll:.4f}")

Train accuracy: 0.5974 | Train log‑loss: 0.6610
Test  accuracy: 0.5080 | Test  log‑loss: 0.7310


### Choix BT with HCA

In [6]:
def estimate_home_bias(df_train, s_hat, team_to_idx):
    """Fit logistic regression to estimate home bias h = log(theta)."""
    # (1) Predictor: team strength difference
    X = np.array([s_hat[team_to_idx[h]] - s_hat[team_to_idx[a]]
                  for h, a in zip(df_train.home_abbr, df_train.away_abbr)])
    y = df_train.home_win.astype(int).values

    # (2) Fit logistic regression with intercept only (via statsmodels)
    X_with_const = sm.add_constant(X)
    model = sm.Logit(y, X_with_const).fit(disp=False)

    h_hat = model.params[0]        # intercept → home‑advantage bias
    print(model.summary())         # detailed regression summary

    print(f"\nEstimated home effect: h = {h_hat:.3f}  →  θ = {np.exp(h_hat):.3f}")
    return h_hat

def predict_bt_home(df, s, team_to_idx, h):
    """Bradley–Terry with global home bias."""
    logits = np.array([
        (s[team_to_idx[h_]] - s[team_to_idx[a_]]) + h
        for h_, a_ in zip(df.home_abbr, df.away_abbr)
    ])
    return 1 / (1 + np.exp(-logits))

In [7]:
# --- 2. Estimate home bias
h_hat = estimate_home_bias(df_train, s_hat, team_to_idx)

# --- 3. Evaluate on both splits
for name, df_split in [("Train", df_train), ("Test", df_test)]:
    y = df_split.home_win.astype(int).values
    p = predict_bt_home(df_split, s_hat, team_to_idx, h_hat)
    acc = accuracy_score(y, (p >= 0.5))
    ll  = log_loss(y, p)
    print(f"{name:5s}  acc: {acc:.4f} |  log‑loss: {ll:.4f}")

                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                 9538
Model:                          Logit   Df Residuals:                     9536
Method:                           MLE   Df Model:                            1
Date:                Sat, 21 Feb 2026   Pseudo R-squ.:                 0.04696
Time:                        14:23:32   Log-Likelihood:                -6200.2
converged:                       True   LL-Null:                       -6505.7
Covariance Type:            nonrobust   LLR p-value:                6.685e-135
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.3078      0.021     14.375      0.000       0.266       0.350
x1             1.0218      0.046     22.403      0.000       0.932       1.111

Estimated home effect: h = 0.308  →  θ = 1.360
Trai